# Eksplorasi KB Hukum Ketenagakerjaan

Notebook ini meniru **DL Query** dan **SPARQL** dari Protege memakai Python, supaya bisa
melihat hasil inferensi tanpa bergantung pada plugin SPARQL Protege (yang error karena
bundle Guava tidak lengkap).

Notebook memakai **KB murni** (`core`, `meta`, `regulasi`) tanpa fixture contoh.
Karena KB tidak berisi individu orang, kita membuat kasus sendiri di bagian 3
(seperti input form), lalu reasoner menyimpulkan haknya.

- **DL Query** di Protege = daftar individu yang memenuhi suatu kelas/ekspresi.
  Padanannya: `Kelas.instances()` setelah reasoning (kelas bernama), atau filter Python.
- **SPARQL**:
  - `rdflib` = data tersurat saja (Turtle yang kita tulis).
  - `owlready2` = setelah `sync_reasoner`, bisa melihat hasil inferensi.


In [1]:
import sys, pathlib

def cari_root(start):
    p = start.resolve()
    for _ in range(8):
        if (p / "scripts" / "check_ontology.py").exists():
            return p
        p = p.parent
    raise RuntimeError("root proyek tidak ditemukan")

ROOT = cari_root(pathlib.Path.cwd())
sys.path.insert(0, str(ROOT / "scripts"))
import check_ontology as kb  # noqa: E402

world = kb.load_reasoned(termasuk_contoh=False)   # KB murni, tanpa fixture
print("KB siap. Cache:", kb.CACHE_KB.name)


[muat] 14 modul Turtle (KB murni), cache kb.rdfxml dipakai


* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /home/prayudahlah/projects/college/kbr/project/prototype1/.venv/lib/python3.12/site-packages/owlready2/hermit:/home/prayudahlah/projects/college/kbr/project/prototype1/.venv/lib/python3.12/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmpojnhy1e9 -Y


KB siap. Cache: kb.rdfxml


* Owlready2 * HermiT took 0.6877095699310303 seconds
* Owlready * Reparenting ketenagakerjaan.berhakAtasUangPisah: {ketenagakerjaan.berhakAtas, owl.ObjectProperty} => {ketenagakerjaan.berhakAtas}
* Owlready * Reparenting ketenagakerjaan.berhakAtasUPMK: {ketenagakerjaan.berhakAtas, owl.ObjectProperty} => {ketenagakerjaan.berhakAtas}
* Owlready * Reparenting ketenagakerjaan.berhakAtasUPH: {ketenagakerjaan.berhakAtas, owl.ObjectProperty} => {ketenagakerjaan.berhakAtas}
* Owlready * Reparenting ketenagakerjaan.berhakAtasPesangon: {ketenagakerjaan.berhakAtas, owl.ObjectProperty} => {ketenagakerjaan.berhakAtas}
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


## 1. Helper: hak per individu

`hak_individu` mengembalikan kelas `MemenuhiSyarat*` yang diinferensikan untuk
seorang individu. Ini padanan DL Query per orang.


In [2]:
KELAS_HAK = [
    "MemenuhiSyaratPesangon", "MemenuhiSyaratUPMK", "MemenuhiSyaratUPH",
    "MemenuhiSyaratUangPisah", "MemenuhiSyaratCutiTahunan", "MemenuhiSyaratCutiHaid",
    "MemenuhiSyaratCutiKhusus", "MemenuhiSyaratUpahLembur",
    "MemenuhiSyaratKompensasiPKWT", "MemenuhiSyaratJaminanSosial",
    "MemenuhiSyaratCutiMelahirkan", "MemenuhiSyaratCutiKeguguran",
    "MemenuhiSyaratMenyusui", "MemenuhiSyaratCutiPendampinganSuami",
]

def hak_individu(nama):
    ind = kb.ktn(nama)
    return [k.replace("MemenuhiSyarat", "") for k in KELAS_HAK
            if ind in kb.ktn(k).instances()]

print("helper siap")


helper siap


## 2. Simulasi input kasus

Membuat beberapa individu seperti input form, lalu reasoner menyimpulkan haknya.
Tidak ada yang ditulis ke berkas KB; ini hanya di memori sesi.


In [3]:
from owlready2 import sync_reasoner, destroy_entity

NAMA_KASUS = ["KasusEfisiensi", "PHKEfisiensi",
              "KasusResign", "PHKResign",
              "KasusMelahirkan"]

# bersihkan bila sel dijalankan ulang
for nama in NAMA_KASUS:
    lama = world.search_one(iri=f"https://example.org/kbr/ketenagakerjaan#{nama}")
    if lama is not None:
        destroy_entity(lama)

Pekerja = kb.ktn("Pekerja")
PekerjaPerempuan = kb.ktn("PekerjaPerempuan")
PHK = kb.ktn("PHK")

# Kasus A: di-PHK karena efisiensi rugi, masa kerja 10 tahun
a = Pekerja("KasusEfisiensi")
a.masaKerjaBulan = [120]
a.upahBulanan = [5000000]
phk_a = PHK("PHKEfisiensi")
phk_a.memilikiAlasanPHK = kb.ktn("EfisiensiRugi")   # functional: nilai tunggal
a.mengalami = [phk_a]

# Kasus B: mengundurkan diri
b = Pekerja("KasusResign")
b.masaKerjaBulan = [60]
b.upahBulanan = [4000000]
phk_b = PHK("PHKResign")
phk_b.memilikiAlasanPHK = kb.ktn("MengundurkanDiri")
b.mengalami = [phk_b]

# Kasus C: ibu cuti melahirkan 6 bulan
c = PekerjaPerempuan("KasusMelahirkan")
c.masaKerjaBulan = [24]
c.upahBulanan = [5000000]
c.sedangCutiMelahirkan = [True]
c.cutiMelahirkanBulan = [6]

sync_reasoner()

for nama in ["KasusEfisiensi", "KasusResign", "KasusMelahirkan"]:
    print(f"{nama:18s}: {hak_individu(nama)}")


* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /home/prayudahlah/projects/college/kbr/project/prototype1/.venv/lib/python3.12/site-packages/owlready2/hermit:/home/prayudahlah/projects/college/kbr/project/prototype1/.venv/lib/python3.12/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmpdgu3_4c4


KasusEfisiensi    : ['Pesangon', 'UPMK', 'UPH', 'CutiTahunan', 'JaminanSosial']
KasusResign       : ['UPH', 'UangPisah', 'CutiTahunan', 'JaminanSosial']
KasusMelahirkan   : ['CutiTahunan', 'CutiHaid', 'JaminanSosial', 'CutiMelahirkan']


* Owlready2 * HermiT took 0.6909379959106445 seconds
* Owlready * Equivalenting: ketenagakerjaan.MemenuhiSyaratJaminanSosial ketenagakerjaan.Pekerja
* Owlready * Equivalenting: ketenagakerjaan.Pekerja ketenagakerjaan.MemenuhiSyaratJaminanSosial
* Owlready * Equivalenting: ketenagakerjaan.MemenuhiSyaratCutiHaid ketenagakerjaan.PekerjaPerempuan
* Owlready * Equivalenting: ketenagakerjaan.PekerjaPerempuan ketenagakerjaan.MemenuhiSyaratCutiHaid
* Owlready * Reparenting ketenagakerjaan.PekerjaPerempuan: {ketenagakerjaan.Pekerja} => {ketenagakerjaan.MemenuhiSyaratJaminanSosial}
* Owlready * Reparenting ketenagakerjaan.MemenuhiSyaratCutiHaid (since equivalent): {ketenagakerjaan.Pekerja} => {ketenagakerjaan.MemenuhiSyaratJaminanSosial}
* Owlready * Reparenting ketenagakerjaan.MengundurkanDiri: {ketenagakerjaan.AlasanPHK, ketenagakerjaan.AlasanTanpaPesangon} => {ketenagakerjaan.AlasanTanpaPesangon}
* Owlready * Reparenting ketenagakerjaan.KasusMelahirkan: {ketenagakerjaan.PekerjaPerempuan} => {

## 3. Padanan DL Query: instances kelas defined

Setelah kasus dibuat, ini sama dengan mengetik nama kelas di tab DL Query Protege
lalu centang *Instances*.


In [4]:
for nama in KELAS_HAK:
    inst = sorted(i.name for i in kb.ktn(nama).instances())
    print(f"{nama:35s} -> {inst or '(kosong)'}")


MemenuhiSyaratPesangon              -> ['KasusEfisiensi']
MemenuhiSyaratUPMK                  -> ['KasusEfisiensi']
MemenuhiSyaratUPH                   -> ['KasusEfisiensi', 'KasusResign']
MemenuhiSyaratUangPisah             -> ['KasusResign']
MemenuhiSyaratCutiTahunan           -> ['KasusEfisiensi', 'KasusMelahirkan', 'KasusResign']
MemenuhiSyaratCutiHaid              -> ['KasusMelahirkan']
MemenuhiSyaratCutiKhusus            -> (kosong)
MemenuhiSyaratUpahLembur            -> (kosong)
MemenuhiSyaratKompensasiPKWT        -> (kosong)
MemenuhiSyaratJaminanSosial         -> ['KasusEfisiensi', 'KasusMelahirkan', 'KasusResign']
MemenuhiSyaratCutiMelahirkan        -> ['KasusMelahirkan']
MemenuhiSyaratCutiKeguguran         -> (kosong)
MemenuhiSyaratMenyusui              -> (kosong)
MemenuhiSyaratCutiPendampinganSuami -> (kosong)


## 4. Padanan DL Query: ekspresi

Di Protege: `Pekerja and (masaKerjaBulan some xsd:integer[>= 12])`.
owlready2 tidak mem-parse sintaks Manchester, jadi ditulis sebagai filter Python.


In [5]:
Pekerja = kb.ktn("Pekerja")
layak = sorted(p.name for p in Pekerja.instances()
               if p.masaKerjaBulan and p.masaKerjaBulan[0] >= 12)
print("Pekerja dengan masa kerja >= 12 bulan:", layak)


Pekerja dengan masa kerja >= 12 bulan: ['KasusEfisiensi', 'KasusMelahirkan', 'KasusResign']


## 5. SPARQL atas data tersurat (rdflib)

Query di bawah berjalan atas graph Turtle yang kita tulis, tanpa inferensi.


In [6]:
import rdflib

g = rdflib.Graph()
g.parse(kb.CACHE_KB, format="xml")

PREFIX = """
PREFIX ktn: <https://example.org/kbr/ketenagakerjaan#>
PREFIX reg: <https://example.org/kbr/regulasi#>
PREFIX dct: <http://purl.org/dc/terms/>
"""

def jalankan(judul, query):
    print(f"== {judul} ==")
    for row in g.query(PREFIX + query):
        print("  ", " | ".join(str(c) for c in row))
    print()

jalankan("Alasan PHK dan pengalinya", """
SELECT ?alasan ?up ?upmk WHERE {
  ?alasan a ktn:AlasanPHK ; ktn:pengaliUP ?up ; ktn:pengaliUPMK ?upmk .
} ORDER BY ?up LIMIT 25
""")

jalankan("Alasan tanpa pesangon (pengaliUP = 0)", """
SELECT ?alasan WHERE {
  ?alasan a ktn:AlasanPHK ; ktn:pengaliUP ?up .
  FILTER(?up = 0)
}
""")

jalankan("Tarif iuran jaminan sosial", """
SELECT ?program ?jenis ?tarif WHERE {
  ?program ktn:jenisProgram ?jenis ; ktn:tarifTotalPersen ?tarif .
} ORDER BY ?jenis ?tarif
""")

jalankan("Cuti khusus dan jumlah hari", """
SELECT ?peristiwa ?hari WHERE {
  ?peristiwa a ktn:PeristiwaCutiKhusus ; ktn:hariDibayar ?hari .
} ORDER BY ?hari
""")


== Alasan PHK dan pengalinya ==
   https://example.org/kbr/ketenagakerjaan#MengundurkanDiri | 0 | 0
   https://example.org/kbr/ketenagakerjaan#PutusanPPHITidakBersalah | 0 | 0
   https://example.org/kbr/ketenagakerjaan#DitahanTidakMerugikan | 0 | 1
   https://example.org/kbr/ketenagakerjaan#PelanggaranMendesak | 0 | 0
   https://example.org/kbr/ketenagakerjaan#Mangkir | 0 | 0
   https://example.org/kbr/ketenagakerjaan#DitahanMerugikan | 0 | 0
   https://example.org/kbr/ketenagakerjaan#EfisiensiRugi | 0.5 | 1
   https://example.org/kbr/ketenagakerjaan#PengambilalihanUbahSyarat | 0.5 | 1
   https://example.org/kbr/ketenagakerjaan#PKPUKarenaRugi | 0.5 | 1
   https://example.org/kbr/ketenagakerjaan#ForceMajeureTutup | 0.5 | 1
   https://example.org/kbr/ketenagakerjaan#TutupKarenaRugi | 0.5 | 1
   https://example.org/kbr/ketenagakerjaan#Pailit | 0.5 | 1
   https://example.org/kbr/ketenagakerjaan#PelanggaranSP | 0.5 | 1
   https://example.org/kbr/ketenagakerjaan#ForceMajeureTidakTutup | 0.75

## 6. SPARQL dengan hasil inferensi (owlready2)

Setelah `sync_reasoner`, owlready2 bisa menjawab keanggotaan kelas hasil inferensi.
Contoh di bawah menjawab "siapa yang berhak pesangon" dari kasus yang kita buat.

Catatan: untuk kelas yang didefinisikan sebagai `= kelas bernama` (equivalen),
mis. `MemenuhiSyaratJaminanSosial = Pekerja`, SPARQL owlready2 bisa melaporkan
lebih sedikit individu daripada `instances()`. Untuk kasus seperti itu pakai
`instances()` (bagian 1/3).


In [7]:
from owlready2 import default_world

def sparql_inferensi(query):
    return [[str(c).split(".")[-1] for c in row]
            for row in default_world.sparql(PREFIX + query)]

print("MemenuhiSyaratPesangon ->", sparql_inferensi("""
SELECT ?x WHERE { ?x a ktn:MemenuhiSyaratPesangon }
"""))

print("MemenuhiSyaratCutiTahunan ->", sparql_inferensi("""
SELECT ?x WHERE { ?x a ktn:MemenuhiSyaratCutiTahunan }
"""))


MemenuhiSyaratPesangon -> [['KasusEfisiensi']]
MemenuhiSyaratCutiTahunan -> [['KasusEfisiensi'], ['KasusResign'], ['KasusMelahirkan']]


## Catatan

- Kapan pakai mana:
  - `Kelas.instances()` (owlready2): paling andal, untuk kelas bernama.
  - `default_world.sparql` (owlready2): SPARQL termasuk hasil inferensi, kecuali kelas equivalen.
  - `rdflib` SPARQL: query data tersurat, tidak perlu reasoning, tidak butuh Java.
- Nominal (pesangon, lembur, kompensasi) dihitung di Python, contohnya di
  `scripts/check_ontology.py`.
- Confidence tidak muncul di sini karena memang tidak masuk Turtle.
- Pelajaran owlready2: object property `FunctionalProperty` diisi nilai tunggal
  (`phk.memilikiAlasanPHK = EfisiensiRugi`), bukan list. Data property dan object
  property biasa memakai list.
